# GS CitizenTech · Vésper — Pipeline Big Data
**Disciplina:** Big Data · **Produto:** Vésper (ruas escuras) · **Lakehouse:** Databricks Free Edition

Arquitetura: `Fontes (SP156 · Open-Meteo · IBGE) → Airflow → chegada/ → Volume → Bronze → Prata → Ouro → MVP`

Ordem de execução: rode as células de cima para baixo. Pré-requisito: os arquivos já no Volume `/Volumes/workspace/gs/chegada` (lotes do Airflow).

## Bronze — dado bruto + colunas de rastro
`bronze_sp156` (xlsx via pandas) · `bronze_clima` e `bronze_ibge` (JSON via COPY INTO, incremental e idempotente).

In [0]:
# openpyxl p/ ler xlsx — em celula separada pois restartPython reinicia o kernel
%pip install openpyxl -q
dbutils.library.restartPython()

In [0]:
# bronze_sp156 — SP156 (xlsx) -> Bronze
import pandas as pd, glob, unicodedata
from pyspark.sql.functions import current_timestamp

BASE = "/Volumes/workspace/gs/chegada"
arquivos = sorted(glob.glob(f"{BASE}/sp156_*.xlsx"))

frames = []
for a in arquivos:
    df = pd.read_excel(a, sheet_name="Solicitações")
    df["arquivo_origem"] = a.split("/")[-1]
    frames.append(df)
pdf = pd.concat(frames, ignore_index=True)

def limpa(c):
    c = unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode()
    return c.strip().lower().replace(" ", "_")
pdf.columns = [limpa(c) for c in pdf.columns]
pdf = pdf.astype(str)

sdf = spark.createDataFrame(pdf).withColumn("carregado_em", current_timestamp())
sdf.write.mode("overwrite").saveAsTable("workspace.gs.bronze_sp156")
print("bronze_sp156 linhas:", sdf.count())

bronze_sp156 linhas: 968464


In [0]:
# bronze_clima e bronze_ibge via COPY INTO (idempotente + incremental)
spark.sql("CREATE TABLE IF NOT EXISTS workspace.gs.bronze_clima")
spark.sql("""
COPY INTO workspace.gs.bronze_clima
FROM (SELECT *, _metadata.file_name AS arquivo_origem
      FROM '/Volumes/workspace/gs/chegada')
FILEFORMAT = JSON PATTERN = 'clima_*.json'
FORMAT_OPTIONS ('multiLine' = 'true') COPY_OPTIONS ('mergeSchema' = 'true')
""")

spark.sql("CREATE TABLE IF NOT EXISTS workspace.gs.bronze_ibge")
spark.sql("""
COPY INTO workspace.gs.bronze_ibge
FROM (SELECT *, _metadata.file_name AS arquivo_origem
      FROM '/Volumes/workspace/gs/chegada')
FILEFORMAT = JSON PATTERN = 'municipios_sp_*.json'
FORMAT_OPTIONS ('multiLine' = 'true') COPY_OPTIONS ('mergeSchema' = 'true')
""")

print("bronze_clima linhas:", spark.table("workspace.gs.bronze_clima").count())
print("bronze_ibge  linhas:", spark.table("workspace.gs.bronze_ibge").count())

bronze_clima linhas: 2
bronze_ibge  linhas: 1290


## Prata — tipos, nulos/inválidos, MERGE pela chave, 3 regras de qualidade

In [0]:
# prata_sp156 — limpeza + tipos + chave de negocio + MERGE (upsert idempotente)
from pyspark.sql import functions as F
from delta.tables import DeltaTable

b = spark.table("workspace.gs.bronze_sp156")

# 1) "nan"/"NaT"/vazio -> null
lixo = ["nan","NaN","NAN","None","NaT","nat","NAT",""]
s = b.select([
    F.when(F.trim(F.col(c)).isin(lixo), None).otherwise(F.trim(F.col(c))).alias(c)
    for c in b.columns])

# 2) tipos corretos (try_* = null em vez de erro)
s = (s
    .withColumn("data_de_abertura",    F.expr("try_to_timestamp(data_de_abertura)"))
    .withColumn("data_de_finalizacao", F.expr("try_to_timestamp(data_de_finalizacao)"))
    .withColumn("latitude",  F.expr("try_cast(latitude as double)"))
    .withColumn("longitude", F.expr("try_cast(longitude as double)"))
    .withColumn("cep", F.regexp_replace(F.col("cep"), "[^0-9]", "")))

# 3) coordenadas fora da caixa de SP -> null
s = (s
    .withColumn("latitude",  F.when(F.col("latitude").between(-24.1, -23.3),  F.col("latitude")))
    .withColumn("longitude", F.when(F.col("longitude").between(-47.0, -46.3), F.col("longitude"))))

# 4) chave de negocio (SP156 nao tem ID -> hash das colunas estaveis)
s = s.withColumn("id_chamado", F.sha2(F.concat_ws("|",
        F.coalesce(F.col("data_de_abertura").cast("string"), F.lit("")),
        F.coalesce(F.col("assunto"),   F.lit("")),
        F.coalesce(F.col("servico"),   F.lit("")),
        F.coalesce(F.col("logradouro"),F.lit("")),
        F.coalesce(F.col("numero"),    F.lit("")),
        F.coalesce(F.col("distrito"),  F.lit(""))), 256))

# 5) dedup pela chave + carimbo
s = s.dropDuplicates(["id_chamado"]).withColumn("processado_em", F.current_timestamp())

# 6) MERGE pela chave
if not spark.catalog.tableExists("workspace.gs.prata_sp156"):
    s.limit(0).write.saveAsTable("workspace.gs.prata_sp156")
tgt = DeltaTable.forName(spark, "workspace.gs.prata_sp156")
(tgt.alias("t").merge(s.alias("s"), "t.id_chamado = s.id_chamado")
    .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

print("bronze_sp156 linhas:", b.count())
print("prata_sp156  linhas:", spark.table("workspace.gs.prata_sp156").count())

bronze_sp156 linhas: 968464
prata_sp156  linhas: 277009


In [0]:
# 3 regras de qualidade com contagem (ok x falhas)
from pyspark.sql import functions as F
p = spark.table("workspace.gs.prata_sp156")
total = p.count()
r1_ok = p.filter(F.col("data_de_abertura").isNotNull()).count()
r2_ok = p.filter(F.col("latitude").isNotNull() & F.col("longitude").isNotNull()).count()
r3_ok = p.filter(F.col("status").isNotNull() & F.col("distrito").isNotNull()).count()
print(f"total linhas: {total}\n")
print(f"R1 | data_de_abertura preenchida  : {r1_ok:>7} ok | {total-r1_ok:>7} falha")
print(f"R2 | coordenadas validas (caixa SP): {r2_ok:>7} ok | {total-r2_ok:>7} falha")
print(f"R3 | status + distrito preenchidos : {r3_ok:>7} ok | {total-r3_ok:>7} falha")

total linhas: 277009

R1 | data_de_abertura preenchida  :  277009 ok |       0 falha
R2 | coordenadas validas (caixa SP):  264275 ok |   12734 falha
R3 | status + distrito preenchidos :  264195 ok |   12814 falha


## Ouro — modelo estrela (fato + 2 dimensões) e as 5 consultas

In [0]:
# ref_populacao_distrito — populacao por distrito (IBGE Censo 2022, agregado 9923)
import requests, unicodedata
from collections import defaultdict
from pyspark.sql import functions as F

url = ("https://servicodados.ibge.gov.br/api/v3/agregados/9923"
       "/periodos/2022/variaveis?localidades=N10[N6[3550308]]")
dados = requests.get(url, timeout=120).json()
var = dados[0]

tem_total, total_por_distrito, soma_por_distrito = False, {}, defaultdict(int)
for res in var["resultados"]:
    cats = res.get("classificacoes", [])
    cat_nome = list(cats[0]["categoria"].values())[0] if cats else "Total"
    for sr in res["series"]:
        nome = sr["localidade"]["nome"]
        raw  = sr["serie"].get("2022")
        val  = int(raw) if raw not in (None, "...", "-", "") else 0
        if cat_nome.strip().lower() == "total":
            tem_total = True; total_por_distrito[nome] = val
        else:
            soma_por_distrito[nome] += val
final = total_por_distrito if tem_total else dict(soma_por_distrito)

def norm(x):
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode()
    return x.strip().lower()

linhas = [(nome, pop, norm(nome)) for nome, pop in final.items()]
ref = spark.createDataFrame(linhas, ["distrito_ibge", "populacao", "distrito_key"])
ref.write.mode("overwrite").saveAsTable("workspace.gs.ref_populacao_distrito")
print("distritos:", ref.count(),
      "| pop total cidade:", ref.agg(F.sum("populacao")).collect()[0][0])

distritos: 96 | pop total cidade: 11451999


In [0]:
# dim_local — distrito + zona (Centro/Periferia) + populacao
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType
import unicodedata

def norm(x):
    if x is None: return None
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode()
    return x.strip().lower()
norm_udf = F.udf(norm)

# Centro = Subprefeitura da Se (8 distritos centrais); resto = Periferia
centro = ["se","republica","bela vista","bom retiro","cambuci",
          "consolacao","liberdade","santa cecilia"]

ref = (spark.table("workspace.gs.ref_populacao_distrito")
       .withColumn("distrito_key", norm_udf(F.split(F.col("distrito_ibge"), " - ").getItem(0)))
       .select("distrito_key",
               F.split(F.col("distrito_ibge"), " - ").getItem(0).alias("distrito"),
               F.col("populacao").cast("long").alias("populacao")))

dim_local = (ref
    .withColumn("zona", F.when(F.col("distrito_key").isin(centro), "Centro").otherwise("Periferia"))
    .withColumn("local_id", F.sha2(F.col("distrito_key"), 256))
    .select("local_id","distrito_key","distrito","zona","populacao"))

sch = StructType([
    StructField("local_id", StringType()), StructField("distrito_key", StringType()),
    StructField("distrito", StringType()), StructField("zona", StringType()),
    StructField("populacao", LongType())])
sentinela = spark.createDataFrame(
    [("NAO_INFORMADO","__nao_informado__","Não informado","Não informado", None)], sch)

dim_local = dim_local.unionByName(sentinela)
dim_local.write.mode("overwrite").saveAsTable("workspace.gs.dim_local")
print("linhas dim_local:", dim_local.count())
dim_local.groupBy("zona").agg(F.count("*").alias("distritos"),
                              F.sum("populacao").alias("populacao")).orderBy("zona").show()

linhas dim_local: 97
+-------------+---------+---------+
|         zona|distritos|populacao|
+-------------+---------+---------+
|       Centro|        8|   423536|
|Não informado|        1|     NULL|
|    Periferia|       88| 11028463|
+-------------+---------+---------+



In [0]:
# dim_tempo — a partir da data_de_abertura
from pyspark.sql import functions as F
p = spark.table("workspace.gs.prata_sp156")
meses = ["jan","fev","mar","abr","mai","jun","jul","ago","set","out","nov","dez"]
dias  = ["dom","seg","ter","qua","qui","sex","sab"]

dim_tempo = (p.select(F.to_date("data_de_abertura").alias("data"))
    .filter(F.col("data").isNotNull()).distinct()
    .withColumn("id_tempo",   F.date_format("data","yyyyMMdd").cast("int"))
    .withColumn("ano",        F.year("data"))
    .withColumn("trimestre",  F.quarter("data"))
    .withColumn("mes",        F.month("data"))
    .withColumn("nome_mes",   F.element_at(F.array(*[F.lit(m) for m in meses]), F.month("data")))
    .withColumn("dia",        F.dayofmonth("data"))
    .withColumn("dia_semana", F.element_at(F.array(*[F.lit(d) for d in dias]), F.dayofweek("data")))
    .select("id_tempo","data","ano","trimestre","mes","nome_mes","dia","dia_semana"))
dim_tempo.write.mode("overwrite").saveAsTable("workspace.gs.dim_tempo")
print("linhas dim_tempo:", dim_tempo.count())

linhas dim_tempo: 92


In [0]:
# fato_chamado — grao = 1 chamado; FKs para as 2 dimensoes + medidas
from pyspark.sql import functions as F
import unicodedata

def norm(x):
    if x is None: return None
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode()
    return x.strip().lower()
norm_udf = F.udf(norm)

p  = spark.table("workspace.gs.prata_sp156")
dl = spark.table("workspace.gs.dim_local").select("local_id", "distrito_key")

f = (p
    .withColumn("distrito_key", norm_udf(F.col("distrito")))
    .withColumn("id_tempo", F.date_format(F.to_date("data_de_abertura"), "yyyyMMdd").cast("int"))
    .withColumn("tempo_conserto_dias",
        F.when(F.col("data_de_abertura").isNotNull() & F.col("data_de_finalizacao").isNotNull(),
               F.datediff(F.to_date("data_de_finalizacao"), F.to_date("data_de_abertura"))))
    .withColumn("flag_finalizado",
        F.when(F.lower(F.col("status")).contains("finaliz") |
               F.col("data_de_finalizacao").isNotNull(), 1).otherwise(0)))

f = (f.join(dl, "distrito_key", "left")
       .withColumn("local_id", F.coalesce(F.col("local_id"), F.lit("NAO_INFORMADO"))))
f = f.withColumn("flag_valido",
        F.when((F.col("local_id") != "NAO_INFORMADO") &
               F.col("data_de_abertura").isNotNull(), 1).otherwise(0))

fato = f.select("id_chamado","id_tempo","local_id","status",
                "data_de_abertura","data_de_finalizacao",
                "tempo_conserto_dias","flag_finalizado","flag_valido")
fato.write.mode("overwrite").saveAsTable("workspace.gs.fato_chamado")
print("linhas fato_chamado:", fato.count())
print("validos:", fato.filter("flag_valido=1").count(),
      "| descartados:", fato.filter("flag_valido=0").count())

linhas fato_chamado: 277009
validos: 262915 | descartados: 14094


### As 5 consultas SQL

In [0]:
# Q1 - Chamados por distrito (onde a cidade esta no escuro)
spark.sql("""
SELECT l.distrito, l.zona, COUNT(*) AS chamados
FROM workspace.gs.fato_chamado f
JOIN workspace.gs.dim_local l ON f.local_id = l.local_id
WHERE l.zona <> 'Não informado'
GROUP BY l.distrito, l.zona
ORDER BY chamados DESC LIMIT 10
""").show(truncate=False)

+---------------+---------+--------+
|distrito       |zona     |chamados|
+---------------+---------+--------+
|Jabaquara      |Periferia|6283    |
|Grajaú         |Periferia|5439    |
|Itaquera       |Periferia|5046    |
|Santana        |Periferia|5004    |
|Capão Redondo  |Periferia|4641    |
|República      |Centro   |4538    |
|Santo Amaro    |Periferia|4413    |
|Sé             |Centro   |4308    |
|Jardim São Luís|Periferia|4299    |
|Cidade Ademar  |Periferia|4297    |
+---------------+---------+--------+



In [0]:
# Q2 - Tempo medio ate o conserto
spark.sql("""
SELECT ROUND(AVG(tempo_conserto_dias),1) AS media_dias,
       percentile(tempo_conserto_dias, 0.5) AS mediana_dias,
       COUNT(*) AS finalizados
FROM workspace.gs.fato_chamado
WHERE flag_finalizado = 1 AND tempo_conserto_dias >= 0
""").show()

+----------+------------+-----------+
|media_dias|mediana_dias|finalizados|
+----------+------------+-----------+
|       6.5|         1.0|     210825|
+----------+------------+-----------+



In [0]:
# Q3 - Evolucao de chamados por mes
spark.sql("""
SELECT t.ano, t.mes, t.nome_mes, COUNT(*) AS chamados
FROM workspace.gs.fato_chamado f
JOIN workspace.gs.dim_tempo t ON f.id_tempo = t.id_tempo
GROUP BY t.ano, t.mes, t.nome_mes
ORDER BY t.ano, t.mes
""").show()

+----+---+--------+--------+
| ano|mes|nome_mes|chamados|
+----+---+--------+--------+
|2026|  7|     jul|   88795|
|2026|  8|     ago|   91939|
|2026|  9|     set|   96275|
+----+---+--------+--------+



In [0]:
# Q4 - % de chamados validos vs descartados
spark.sql("""
SELECT CASE flag_valido WHEN 1 THEN 'valido' ELSE 'descartado' END AS tipo,
       COUNT(*) AS chamados,
       ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (),1) AS pct
FROM workspace.gs.fato_chamado
GROUP BY flag_valido ORDER BY chamados DESC
""").show()

+----------+--------+----+
|      tipo|chamados| pct|
+----------+--------+----+
|    valido|  262915|94.9|
|descartado|   14094| 5.1|
+----------+--------+----+



In [0]:
# Q5 (IMPACTO) - chamados por mil habitantes, Centro x Periferia (junta o fato com IBGE)
spark.sql("""
WITH pop_zona AS (
  SELECT zona, SUM(populacao) AS populacao
  FROM workspace.gs.dim_local WHERE zona <> 'Não informado'
  GROUP BY zona),
cham_zona AS (
  SELECT l.zona, COUNT(*) AS chamados
  FROM workspace.gs.fato_chamado f
  JOIN workspace.gs.dim_local l ON f.local_id = l.local_id
  WHERE l.zona <> 'Não informado'
  GROUP BY l.zona)
SELECT c.zona, c.chamados, p.populacao,
       ROUND(c.chamados * 1000.0 / p.populacao, 2) AS chamados_por_mil_hab
FROM cham_zona c JOIN pop_zona p ON c.zona = p.zona
ORDER BY chamados_por_mil_hab DESC
""").show()

+---------+--------+---------+--------------------+
|     zona|chamados|populacao|chamados_por_mil_hab|
+---------+--------+---------+--------------------+
|   Centro|   21620|   423536|               51.05|
|Periferia|  241295| 11028463|               21.88|
+---------+--------+---------+--------------------+



## Bloco 7 — LGPD e integração com o MVP
A Ouro entrega **agregados por distrito** (não o indivíduo). O SP156 não traz nome/telefone/e-mail do solicitante; o quase-identificador é o **endereço**, que fica fora da Ouro e é **hasheado** quando necessário (`sha2`). O MVP consome a tabela `ouro_mvp_mapa`.

In [0]:
# Bloco 7 - LGPD: tabela de consumo do MVP (agregada, sem dado pessoal) + demo de hash
from pyspark.sql import functions as F

# 1) Contrato com o produto: tabela agregada que o MVP consome (mapa do cidadao)
mvp = spark.sql("""
SELECT l.distrito, l.zona, l.populacao,
       COUNT(*) AS chamados,
       ROUND(COUNT(*)*1000.0/l.populacao,2) AS chamados_por_mil_hab,
       ROUND(AVG(CASE WHEN f.tempo_conserto_dias>=0 THEN f.tempo_conserto_dias END),1) AS tempo_medio_dias,
       ROUND(100.0*SUM(f.flag_finalizado)/COUNT(*),1) AS pct_finalizado
FROM workspace.gs.fato_chamado f
JOIN workspace.gs.dim_local l ON f.local_id = l.local_id
WHERE l.zona <> 'Não informado'
GROUP BY l.distrito, l.zona, l.populacao
""")
mvp.write.mode("overwrite").saveAsTable("workspace.gs.ouro_mvp_mapa")
print("ouro_mvp_mapa (tabela que o MVP consome):", mvp.count(), "distritos")
mvp.orderBy(F.desc("chamados")).show(5, truncate=False)

# 2) Demonstracao LGPD: hash do endereco (quase-identificador) sobre o dado bruto
protegido = (spark.table("workspace.gs.prata_sp156")
    .withColumn("endereco_hash", F.sha2(F.concat_ws("|",
        F.coalesce(F.col("logradouro"),F.lit("")),
        F.coalesce(F.col("numero"),F.lit("")),
        F.coalesce(F.col("cep"),F.lit(""))), 256))
    .select("id_chamado","distrito","endereco_hash"))
print("\nExemplo de endereco hasheado (LGPD):")
protegido.show(3, truncate=False)

ouro_mvp_mapa (tabela que o MVP consome): 96 distritos
+-------------+---------+---------+--------+--------------------+----------------+--------------+
|distrito     |zona     |populacao|chamados|chamados_por_mil_hab|tempo_medio_dias|pct_finalizado|
+-------------+---------+---------+--------+--------------------+----------------+--------------+
|Jabaquara    |Periferia|214982   |6283    |29.23               |4.8             |83.7          |
|Grajaú       |Periferia|384873   |5439    |14.13               |5.5             |81.1          |
|Itaquera     |Periferia|210960   |5046    |23.92               |5.4             |77.5          |
|Santana      |Periferia|115689   |5004    |43.25               |4.2             |81.0          |
|Capão Redondo|Periferia|270767   |4641    |17.14               |6.3             |75.8          |
+-------------+---------+---------+--------+--------------------+----------------+--------------+
only showing top 5 rows

Exemplo de endereco hasheado (LGPD):
+